In [ ]:
import pandas as pd

df_arapiraca = pd.read_csv(
    "./output/filtered_arapiraca.csv", delimiter=";", low_memory=False
)

In [10]:
display(df_arapiraca)

,LONGITUDE,LATITUDE,NATUREZA_FATO,DATA_HORA,CIDADE_FATO
0,-36.690147,-9.735983,ROUBO A TRANSEUNTE,2022-01-01 03:00:00,Arapiraca
1,-36.655517,-9.785501,ROUBO A TRANSEUNTE,2022-01-01 20:00:00,Arapiraca
2,-36.591520,-9.762967,ROUBO DE VEÍCULO (DE PASSEIO),2022-01-02 09:00:00,Arapiraca
3,-36.664139,-9.768197,ROUBO A TRANSEUNTE,2022-01-02 23:00:00,Arapiraca
4,-36.680604,-9.644294,ROUBO A TRANSEUNTE,2022-01-03 08:00:00,Arapiraca
...,...,...,...,...,...
19192,-36.662065,-9.764001,ROUBO DE VEÍCULO (MOTO),2014-12-21 01:30:00,Arapiraca
19193,-36.666260,-9.753766,ROUBO A TRANSEUNTE,2014-12-22 14:30:00,Arapiraca
19194,-36.663400,-9.748675,ROUBO A TRANSEUNTE,2014-12-22 22:00:00,Arapiraca
19195,-36.648386,-9.721165,ROUBO DE VEÍCULO (MOTO),2014-12-22 23:30:00,Arapiraca


In [12]:
import osmnx as ox
import geopandas as gpd

place = "Arapiraca, Alagoas, Brazil"
print(f"Querying OSM for: {place}")

gdf_place = ox.geocode_to_gdf(place)
polys = gdf_place[gdf_place.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

if polys.empty:
    # fallback: fetch administrative boundaries and pick candidates
    tags = {"boundary": "administrative"}
    adm = ox.geometries_from_place(place, tags=tags)
    polys = adm[adm.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

if polys.empty:
    raise SystemExit(
        "Could not find polygon for Arapiraca on OSM. Check query or network."
    )

# # compute area in metric projection to pick the largest polygon
polys = polys.to_crs("EPSG:3857")
polys["area_m2"] = polys.geometry.area
largest_idx = polys["area_m2"].idxmax()
poly = polys.loc[largest_idx].geometry
# convert back to WGS84
poly = gpd.GeoSeries([poly], crs="EPSG:3857").to_crs("EPSG:4326").iloc[0]

osm_arapiraca = gpd.GeoDataFrame(
    {"name": ["Arapiraca"], "geometry": [poly]}, crs="EPSG:4326"
)
osm_path = "./output/osm_arapiraca.geojson"
osm_arapiraca.to_file(osm_path, driver="GeoJSON")
print(f"Saved Arapiraca polygon to {osm_path}")

Querying OSM for: Arapiraca, Alagoas, Brazil
Saved Arapiraca polygon to ./output/osm_arapiraca.geojson


In [ ]:
# Spatial join points with municipality polygons (geojs-27-mun.json)
# Requires: geopandas
import geopandas as gpd
import pandas as pd

# load municipalities and ensure CRS is EPSG:4326
mun = gpd.read_file("./output/osm_arapiraca.geojson")
mun = mun.to_crs("EPSG:4326")

# create GeoDataFrame for Arapiraca points
gdf_arapiraca = gpd.GeoDataFrame(
    df_arapiraca.copy(),
    geometry=gpd.points_from_xy(df_arapiraca["LONGITUDE"], df_arapiraca["LATITUDE"]),
    crs="EPSG:4326",
)

# spatial join: attach municipality 'name' to each point (left join)
# use predicate='within' so points must fall inside polygon
joined = gpd.sjoin(
    gdf_arapiraca, mun[["name", "geometry"]], how="left", predicate="within"
)

# boolean: specifically in Arapiraca municipality
joined["in_arapiraca"] = joined["name"] == "Arapiraca"

# summary counts
total = len(df_arapiraca)
inside_count = int(joined["in_arapiraca"].sum())
outside_count = total - inside_count
print(f"Arapiraca records total: {total}")
print(f" - inside Arapiraca polygon: {inside_count}")
print(f" - outside Arapiraca polygon: {outside_count}")

# Filter df_arapiraca to only records whose points lie inside the Arapiraca polygon
inside_maceio = joined.loc[joined["in_arapiraca"]].copy()

inside_maceio_clean = inside_maceio.drop(
    columns=["geometry", "name", "in_arapiraca", "index_right"]
)

# reset index and overwrite df_arapiraca with filtered rows
df_arapiraca = inside_maceio_clean.reset_index(drop=True)

# save filtered CSV
output_path = "./output/filtered_arapiraca_inside_polygon.csv"
df_arapiraca.to_csv(output_path, index=False, sep=";")
print(f"Saved {len(df_arapiraca)} Arapiraca rows inside polygon to {output_path}")

Arapiraca records total: 19197
 - inside Arapiraca polygon: 19141
 - outside Arapiraca polygon: 56
Saved 19141 Arapiraca rows inside polygon to ./output/filtered_arapiraca_inside_polygon.csv


In [ ]:
import folium

# Show the map with the polygon and the points inside/outside the polygon
osm = gpd.read_file("./output/osm_arapiraca.geojson").to_crs("EPSG:4326")
centroid = osm.geometry.centroid.iloc[0]
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=11)

folium.GeoJson(
    osm.to_json(),
    name="Arapiraca (OSM)",
    style_function=lambda f: {
        "fillColor": "transparent",
        "color": "blue",
        "weight": 2,
        "opacity": 0.8,
    },
).add_to(m)

# optional: overlay sample points if gdf_maceio exists
for _, r in joined.iterrows():
    folium.CircleMarker(
        [r.geometry.y, r.geometry.x],
        radius=2,
        color="green" if r["in_arapiraca"] else "red",
        fill=True,
    ).add_to(m)
m.save("./output/arapiraca_and_points.html")
print("Saved arapiraca_osm_polygon_map.html")

/tmp/ipykernel_16556/861210788.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = osm.geometry.centroid.iloc[0]


KeyError: 'in_maceio'